# ZZIGNAL — Análisis Odiseo 83
## Notebook para Google Colab
Cargá un CSV de sesión y analizá el impacto de cada variable en las operaciones.

In [ ]:
# @title Cargar CSV (subí tu archivo o usá una ruta)
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import io

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

# Opción A: subir archivo
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Opción B: ruta directa
# filename = '/content/session_995_hft.csv'

print(f'Archivo: {filename}')

In [ ]:
# @title 1. Leer metadata y cargar datos
meta = {}
header_line = 0
with open(filename) as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line.startswith('# session_id='):
            meta['session_id'] = int(line.split('=')[1])
        elif line.startswith('# btc_price_start='):
            meta['btc_start'] = float(line.split('=')[1])
        elif line.startswith('# btc_price_end='):
            meta['btc_end'] = float(line.split('=')[1])
        elif line.startswith('# outcome_result='):
            meta['outcome'] = line.split('=')[1]
        elif line.startswith('# build_version='):
            meta['build'] = line.split('=')[1]
        elif line.startswith('# tick_count='):
            meta['ticks'] = int(line.split('=')[1])
        elif line.startswith('time,'):
            header_line = i + 1
            break

df = pd.read_csv(filename, skiprows=header_line - 1)
print(f'Sesión #{meta["session_id"]} | Build {meta.get("build","?")}')
print(f'BTC: ${meta["btc_start"]:,.0f} → ${meta["btc_end"]:,.0f} ({meta["btc_end"]-meta["btc_start"]:+,.0f}) | Outcome: {meta["outcome"]}')
print(f'Ticks: {len(df)} | Columnas: {len(df.columns)}')
print(f'\nColumnas del dataset:')
for i, col in enumerate(df.columns):
    print(f'  {i+1:>3}. {col}')
df.head(3)

In [ ]:
# @title BTC vs UP/DOWN + Triggers
# Ensure ts column exists
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['time'], errors='coerce')
    df = df.dropna(subset=['ts'])
    df['minute'] = df['ts'].dt.floor('min')
    print(f'ts column recreated, rows: {len(df)}')
# @title BTC vs UP/DOWN + Triggers
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Panel 1: BTC price
ax = axes[0]
ax.plot(df['ts'], df['binance_price'], 'b-', alpha=0.8, lw=0.8, label='BTC USD')
ax.set_ylabel('BTC USD', color='blue')
ax.legend(loc='upper left')
sid = meta.get('session_id','?')
out = meta.get('outcome','?')
ax.set_title(f'Sesión #{sid} — BTC vs UP/DOWN tokens')
ax.grid(True, alpha=0.3)

# Panel 2: UP/DOWN token prices
ax = axes[1]
ax.scatter(df['ts'], df['clob_trade_up'], s=2, c='green', alpha=0.4, label='UP token')
ax.scatter(df['ts'], df['clob_trade_dn'], s=2, c='red', alpha=0.4, label='DOWN token')
ax.axhline(0.83, color='orange', ls='--', alpha=0.6, lw=1, label='Entry 0.83')
ax.axhline(0.97, color='lime', ls=':', alpha=0.5, lw=0.8, label='TP 0.97')
ax.axhline(0.81, color='darkred', ls=':', alpha=0.5, lw=0.8, label='SL 0.81')
ax.set_ylabel('Token Price')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper left', markerscale=3, fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 3: Triggers
ax = axes[2]
trig_up = df[df['clob_trade_up'] >= 0.83]
trig_dn = df[df['clob_trade_dn'] >= 0.83]
ax.scatter(trig_up['ts'], [1]*len(trig_up), s=15, c='green', marker='^', alpha=0.7, label=f'UP ({len(trig_up)})')
ax.scatter(trig_dn['ts'], [-1]*len(trig_dn), s=15, c='red', marker='v', alpha=0.7, label=f'DN ({len(trig_dn)})')
ax.set_ylim(-2, 2)
ax.set_ylabel('Trigger')
ax.set_xlabel('Time (UTC-5)')
ax.set_yticks([-1, 1])
ax.set_yticklabels(['DOWN', 'UP'])
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Triggers UP: {len(trig_up)} | DN: {len(trig_dn)}')
if len(trig_up)>0:
    print(f'UP avg price: {trig_up["clob_trade_up"].mean():.4f}')
if len(trig_dn)>0:
    print(f'DN avg price: {trig_dn["clob_trade_dn"].mean():.4f}')
entered = (df['od83_up']==2).any() or (df['od83_dn']==2).any()
print(f'Any entry?: {"YES" if entered else "NO (filters blocked)"}')


In [ ]:
# @title 2. Parsear tiempo y agrupar por minuto
df['ts'] = pd.to_datetime(df['time'])
df['minute'] = df['ts'].dt.floor('min')
df['second'] = df['ts'].dt.second + df['ts'].dt.microsecond / 1e6

# Ventanas de 1 minuto
minutes = sorted(df['minute'].unique())
print(f'Rango: {minutes[0]} → {minutes[-1]} ({len(minutes)} ventanas de 1 min)')

# Agrupación por minuto
g = df.groupby('minute')
minute_stats = g.agg(
    ticks=('time', 'count'),
    btc_open=('binance_price', 'first'),
    btc_close=('binance_price', 'last'),
    btc_high=('binance_price', 'max'),
    btc_low=('binance_price', 'min'),
    spread_avg=('spread', lambda x: x[x.between(0, 0.5)].mean()),
    mid_avg=('mid', lambda x: x[x.between(0.001, 0.8)].mean()),
    dump_max=('dump_score', 'max'),
    reversal_max=('reversal_score', 'max'),
    trade_up_avg=('clob_trade_up', lambda x: x[x>0].mean()),
    trade_dn_avg=('clob_trade_dn', lambda x: x[x>0].mean()),
    trade_up_count=('clob_trade_count_up', 'max'),
    trade_dn_count=('clob_trade_count_dn', 'max'),
    od83_up_active=('od83_up', lambda x: (x==2).sum()),
    od83_dn_active=('od83_dn', lambda x: (x==2).sum()),
    triggers_up=('clob_trade_up', lambda x: (x >= 0.83).sum()),
    triggers_dn=('clob_trade_dn', lambda x: (x >= 0.83).sum()),
).reset_index()

minute_stats['btc_delta'] = minute_stats['btc_close'] - minute_stats['btc_open']
print(minute_stats.to_string())

# Visual: BTC por minuto
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# BTC price with triggers
ax = axes[0]
ax.plot(minute_stats['minute'], minute_stats['btc_close'], 'b-o', ms=4, label='BTC')
ax2 = ax.twinx()
ax2.bar(minute_stats['minute'], minute_stats['triggers_up'], width=0.0003, alpha=0.3, color='green', label='Triggers UP')
ax2.bar(minute_stats['minute'], minute_stats['triggers_dn'], width=0.0003, alpha=0.3, color='red', label='Triggers DN')
ax.set_ylabel('BTC USD', color='blue')
ax2.set_ylabel('Triggers', color='gray')
ax.set_title(f'Sesión #{meta["session_id"]} — BTC + Triggers por minuto')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

# Spread + Dump score
ax = axes[1]
ax.plot(minute_stats['minute'], minute_stats['spread_avg'], 'r-o', ms=4, label='Spread avg')
ax2 = ax.twinx()
ax2.plot(minute_stats['minute'], minute_stats['dump_max'], 'k--s', ms=4, label='Dump max')
ax.set_ylabel('Spread', color='red')
ax2.set_ylabel('Dump score', color='black')
ax.set_title('Spread y Dump Score')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

# Trade window
ax = axes[2]
ax.plot(minute_stats['minute'], minute_stats['trade_up_avg'], 'g-o', ms=4, label='trade_up avg')
ax.plot(minute_stats['minute'], minute_stats['trade_dn_avg'], 'r-o', ms=4, label='trade_dn avg')
ax.axhline(0.83, color='orange', ls='--', alpha=0.5, label='Entry threshold 0.83')
ax.set_ylabel('CLOB Trade Price')
ax.set_title('Trade Window Average (señal de entrada)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# @title BTC + Indicadores: ¿cuáles impactan las señales?
# Ensure ts exists
if 'ts' not in df.columns:
    df['ts'] = pd.to_datetime(df['time'], errors='coerce')
    df = df.dropna(subset=['ts'])
# @title BTC + Indicadores: ¿cuáles impactan las señales?
fig, axes = plt.subplots(5, 1, figsize=(16, 14), sharex=True)

# Panel 1: BTC price
ax = axes[0]
ax.plot(df['ts'], df['binance_price'], 'b-', alpha=0.8, lw=0.8, label='BTC')
ax.set_ylabel('BTC USD')
ax.legend(loc='upper left')
ax.set_title(f'Sesión #{meta.get("session_id","?")} — Indicadores vs Señales')
ax.grid(True, alpha=0.3)

# Panel 2: Spread + Dump Score (¿están bien?)
ax = axes[1]
# Solo mostrar spread en ticks con datos reales (no heartbeat mid=0)
real = df[df['mid'].between(0.001, 0.8)]
ax.plot(real['ts'], real['spread'], 'r-', alpha=0.5, lw=0.5, label='Spread')
ax2 = ax.twinx()
ax2.fill_between(df['ts'], 0, df['dump_score'], alpha=0.15, color='red', label='Dump')
ax2.plot(df['ts'], df['dump_score'], 'r-', alpha=0.4, lw=0.5)
ax.set_ylabel('Spread', color='red')
ax2.set_ylabel('Dump Score (0-3)', color='darkred')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(True, alpha=0.3)
# Anotar: spread promedio en ticks reales
avg_spread = real['spread'].mean()
ax.axhline(avg_spread, color='orange', ls='--', alpha=0.4)
ax.text(df['ts'].iloc[len(df)//2], avg_spread, f'  avg spread={avg_spread:.4f}', fontsize=8, color='orange')

# Panel 3: BTC velocity + acceleration
ax = axes[2]
ax.plot(df['ts'], df['btc_vel'], 'purple', alpha=0.5, lw=0.5, label='Vel (USD/s)')
ax.axhline(0, color='gray', ls='-', alpha=0.3)
ax2 = ax.twinx()
ax2.plot(df['ts'], df['btc_acel'], 'orange', alpha=0.3, lw=0.5, label='Acel (USD/s²)')
ax.set_ylabel('Velocity USD/s', color='purple')
ax2.set_ylabel('Accel USD/s²', color='orange')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Panel 4: Trade window count + prices
ax = axes[3]
ax.plot(df['ts'], df['clob_trade_count_up'], 'g-', alpha=0.6, lw=0.5, label='UP count')
ax.plot(df['ts'], df['clob_trade_count_dn'], 'r-', alpha=0.6, lw=0.5, label='DN count')
ax.set_ylabel('Trades in window')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

# Panel 5: Triggers overlay
ax = axes[4]
trig_up = df[df['clob_trade_up'] >= 0.83]
trig_dn = df[df['clob_trade_dn'] >= 0.83]
ax.scatter(trig_up['ts'], [1]*len(trig_up), s=12, c='green', marker='^', alpha=0.6, label=f'UP ({len(trig_up)})')
ax.scatter(trig_dn['ts'], [-1]*len(trig_dn), s=12, c='red', marker='v', alpha=0.6, label=f'DN ({len(trig_dn)})')
ax.set_ylim(-2, 2)
ax.set_ylabel('Trigger')
ax.set_xlabel('Time (UTC-5)')
ax.set_yticks([-1, 1])
ax.set_yticklabels(['DOWN', 'UP'])
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Análisis de impacto
print('=' * 60)
print('  IMPACTO DE INDICADORES EN SEÑALES')
print('=' * 60)
# Cuando hay trigger vs cuando no: ¿qué cambia?
df['is_trigger'] = (df['clob_trade_up'] >= 0.83) | (df['clob_trade_dn'] >= 0.83)
t = df[df['is_trigger']]
nt = df[~df['is_trigger']]
print(f'\n  {"Indicador":<25} {"Con Trigger":>12} {"Sin Trigger":>12} {"Δ":>10}')
print('  ' + '-' * 60)
for col, name in [('btc_vel', 'BTC Vel'), ('btc_acel', 'BTC Acel'),
                   ('spread', 'Spread'), ('dump_score', 'Dump'),
                   ('tick_gap_ms', 'Tick Gap'), ('bid_vol', 'Bid Vol'),
                   ('ask_vol', 'Ask Vol'), ('imbalance', 'Imbalance'),
                   ('reversal_score', 'Reversal'),
                   ('clob_trade_count_up', 'Trade Cnt UP'),
                   ('clob_trade_count_dn', 'Trade Cnt DN')]:
    if col in df.columns:
        tv = t[col].mean()
        nv = nt[col].mean()
        d = tv - nv
        arrow = '↑' if d > 0 else '↓' if d < 0 else '='
        print(f'  {name:<25} {tv:>12.3f} {nv:>12.3f} {arrow} {abs(d):>9.3f}')

print(f'\n  CONCLUSIÓN: Las variables que más cambian cuando hay trigger')
print(f'  son las que más impactan la señal de entrada.')


In [ ]:
# @title 3. Análisis de triggers: ¿qué bloqueó las entradas?
print('=' * 70)
print('  ANÁLISIS DE BLOQUEOS')
print('=' * 70)

# Filtrar ticks con trigger (trade >= 0.83)
trigger = df[(df['clob_trade_up'] >= 0.83) | (df['clob_trade_dn'] >= 0.83)].copy()
print(f'\nTicks con trigger (trade >= 0.83): {len(trigger)}/{len(df)} ({100*len(trigger)/len(df):.1f}%)')

# ¿Por qué no entró? Chequear cada filtro
trigger['block_spread'] = trigger['spread'] > 0.05
trigger['block_dump'] = trigger['dump_score'] >= 2
trigger['block_frozen'] = trigger['tick_gap_ms'] > 3000
trigger['block_low_vol'] = (trigger['bid_vol'] < 500) | (trigger['ask_vol'] < 500)
trigger['block_reversal'] = trigger['reversal_score'] >= 2
trigger['block_secs'] = trigger['secs_left'] <= 45  # last45s boundary

blocks = ['block_spread', 'block_dump', 'block_frozen', 'block_low_vol', 'block_reversal', 'block_secs']
labels = ['Spread>5%', 'Dump>=2', 'Frozen>3s', 'LowVol<500', 'Reversal>=2', 'Last45s']

print(f'\n{"Filtro":<20} {"Bloqueados":>10} {"%":>8}')
print('-' * 40)
for b, l in zip(blocks, labels):
    n = trigger[b].sum()
    print(f'{l:<20} {n:>10} {100*n/len(trigger):>7.1f}%')

# ¿Cuántos ticks NO serían bloqueados por NADIE?
trigger['all_blocked'] = trigger[blocks].any(axis=1)
clean = trigger[~trigger['all_blocked']]
print(f'\n✅ Pasarían todos los filtros: {len(clean)}/{len(trigger)} ({100*len(clean)/max(1,len(trigger)):.1f}%)')
if len(clean) > 0:
    print(f'   Ejemplo: {clean[["time","clob_trade_up","spread","dump_score","tick_gap_ms","bid_vol","ask_vol"]].head(5).to_string()}')

# Visual: waterfall de bloqueos
fig, ax = plt.subplots(figsize=(10, 5))
remaining = len(trigger)
values = [remaining]
names = ['Triggers']
for b, l in zip(blocks, labels):
    blocked = trigger[trigger[b] & ~trigger[blocks[:blocks.index(b)]].any(axis=1)].shape[0]
    remaining -= blocked
    values.append(remaining)
    names.append(f'{l}\n(-{blocked})')

colors = ['blue'] + ['red']*len(blocks)
ax.bar(range(len(values)), values, color=colors, alpha=0.7)
ax.set_xticks(range(len(values)))
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel('Ticks que pasarían')
ax.set_title(f'Waterfall de filtros — {len(trigger)} triggers iniciales → {remaining} sobreviven')
for i, v in enumerate(values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# @title 4. Correlación: ¿qué variables impactan las operaciones?
print('=' * 70)
print('  MATRIZ DE CORRELACIÓN')
print('=' * 70)

# Seleccionar columnas numéricas relevantes para Odiseo
corr_cols = [
    'binance_price', 'btc_vel', 'btc_acel',
    'mid', 'spread', 'bid_vol', 'ask_vol', 'imbalance',
    'clob_trade_up', 'clob_trade_dn', 'clob_trade_count_up', 'clob_trade_count_dn',
    'tick_gap_ms', 'dump_score', 'reversal_score',
    'od83_up', 'od83_dn',
    'mid_from_entry'
]

available = [c for c in corr_cols if c in df.columns]
corr = df[available].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlación — Variables Odiseo 83')
plt.tight_layout()
plt.show()

# Top correlaciones con las señales de entrada
print('\nCorrelación con clob_trade_up (señal de entrada):')
print(corr['clob_trade_up'].sort_values(ascending=False).head(10).to_string())

print('\nCorrelación con od83_up (actividad de trading):')
print(corr['od83_up'].sort_values(ascending=False).head(10).to_string())

In [ ]:
# @title 5. Timeline de la sesión completa
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# Panel 1: BTC price + velocity
ax = axes[0]
ax.plot(df['ts'], df['binance_price'], 'b-', alpha=0.7, lw=0.5, label='BTC')
ax2 = ax.twinx()
ax2.plot(df['ts'], df['btc_vel'], 'orange', alpha=0.5, lw=0.5, label='Vel')
ax.set_ylabel('BTC USD')
ax2.set_ylabel('Vel USD/s', color='orange')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.set_title(f'Sesión #{meta["session_id"]} — BTC ${meta["btc_start"]:,.0f} → ${meta["btc_end"]:,.0f} ({meta["outcome"]})')

# Panel 2: CLOB trade prices
ax = axes[1]
mask = df['clob_trade_up'] > 0
ax.scatter(df.loc[mask, 'ts'], df.loc[mask, 'clob_trade_up'], s=1, c='green', alpha=0.3, label='UP')
mask = df['clob_trade_dn'] > 0
ax.scatter(df.loc[mask, 'ts'], df.loc[mask, 'clob_trade_dn'], s=1, c='red', alpha=0.3, label='DN')
ax.axhline(0.83, color='orange', ls='--', alpha=0.5, label='Entry 0.83')
ax.set_ylabel('CLOB Price')
ax.legend(loc='upper left', markerscale=5)

# Panel 3: Risk scores
ax = axes[2]
ax.fill_between(df['ts'], 0, df['dump_score'], alpha=0.3, color='red', label='dump')
ax.fill_between(df['ts'], 0, df['reversal_score'], alpha=0.3, color='orange', label='reversal')
ax.plot(df['ts'], df['tick_gap_ms'] / 1000, 'k-', alpha=0.3, lw=0.5, label='gap(s)')
ax.set_ylabel('Score')
ax.legend(loc='upper left')

# Panel 4: Trade window count (cuántos trades en la ventana)
ax = axes[3]
ax.plot(df['ts'], df['clob_trade_count_up'], 'g-', alpha=0.7, lw=0.5, label='UP count')
ax.plot(df['ts'], df['clob_trade_count_dn'], 'r-', alpha=0.7, lw=0.5, label='DN count')
ax.set_ylabel('Trades en ventana')
ax.set_xlabel('Tiempo (UTC-5)')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# @title 6. Detalle de entradas/salidas (si las hay)
entries_up = df[df['od83_up'] == 2]
entries_dn = df[df['od83_dn'] == 2]
events = df[df['od83_event'].notna() & (df['od83_event'] != '')]

print(f'Entradas UP: {len(entries_up)} ticks')
print(f'Entradas DN: {len(entries_dn)} ticks')
print(f'Eventos od83_event: {len(events)}')

if len(events) > 0:
    print('\nEventos de entrada/salida:')
    print(events[['time', 'od83_event', 'od83_up_entry', 'od83_up_sz', 'od83_up_pnl', 'od83_up_exit', 'od83_up_r']].to_string())

if len(entries_up) > 0:
    print('\nContexto de entradas UP:')
    cols = ['time', 'clob_trade_up', 'clob_trade_count_up', 'spread', 'dump_score', 'btc_vel', 'btc_acel']
    print(entries_up[cols].head(10).to_string())
    
    # PnL durante posición activa
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(entries_up['ts'], entries_up['od83_up_pnl'], 'g-o', ms=3)
    ax.axhline(0, color='gray', ls='--')
    ax.set_ylabel('PnL Paper ($)')
    ax.set_title('PnL durante posición UP activa')
    plt.show()
else:
    print('\n⚠️  Sin entradas en esta sesión. Revisar waterfall de filtros arriba.')

In [ ]:
# @title Odiseo 83 Performance — Paper Money vs Live
from datetime import timedelta

print('=' * 70)
print('  ODISEO 83 PERFORMANCE TRACKER')
print('=' * 70)

# Mode detection
paper_ticks = df[df['od83_up_mode'] == 'PAPER'] if 'od83_up_mode' in df.columns else df
live_ticks = df[df['od83_up_mode'] == 'LIVE'] if 'od83_up_mode' in df.columns else pd.DataFrame()

mode = 'PAPER' if len(live_ticks) == 0 else 'LIVE'
print(f'\nMode: {mode} | Paper ticks: {len(paper_ticks)} | Live ticks: {len(live_ticks)}')

# Entry/exit events
entries_up = df[df['od83_event'].str.contains('IN_UP', na=False)]
entries_dn = df[df['od83_event'].str.contains('IN_DN', na=False)]
exits_up = df[df['od83_event'].str.contains('OUT_UP', na=False)]
exits_dn = df[df['od83_event'].str.contains('OUT_DN', na=False)]

n_entries = len(entries_up) + len(entries_dn)
n_exits = len(exits_up) + len(exits_dn)

print(f'\nEntries: UP={len(entries_up)} DN={len(entries_dn)} TOTAL={n_entries}')
print(f'Exits:   UP={len(exits_up)} DN={len(exits_dn)} TOTAL={n_exits}')

if n_entries == 0:
    print('\n⚠️  NO ENTRIES — Odiseo did not trade this session')
    print('   Check filter waterfall cell above for reasons')
else:
    # PnL analysis
    up_pnl_final = df.loc[df['od83_up_r'] > 0, ['od83_up_pnl', 'od83_up_r', 'od83_up_entry', 'od83_up_exit']].dropna()
    dn_pnl_final = df.loc[df['od83_dn_r'] > 0, ['od83_dn_pnl', 'od83_dn_r', 'od83_dn_entry', 'od83_dn_exit']].dropna()
    
    print(f'\n{"─"*60}')
    print(f'  TRADE LOG')
    print(f'{"─"*60}')
    print(f'  {"Side":>5} {"Entry":>8} {"Exit":>8} {"Size":>6} {"PnL":>10} {"Reason":>12} {"Result":>8}')
    
    total_pnl = 0
    wins = 0
    losses = 0
    reasons = {1:'TP', 2:'SL-micro', 3:'SL-trend', 4:'SL-hard', 5:'Settle', 6:'Flash'}
    
    for _, r in up_pnl_final.iterrows():
        side = 'UP'
        entry = r['od83_up_entry']
        exit_p = r['od83_up_exit']
        pnl = r['od83_up_pnl']
        reason = int(r['od83_up_r'])
        total_pnl += pnl
        is_win = 'WIN' if pnl > 0 else 'LOSS'
        if pnl > 0: wins += 1
        else: losses += 1
        print(f'  {side:>5} {entry:>8.4f} {exit_p:>8.4f} {7:>6} {pnl:>+10.4f} {reasons.get(reason,str(reason)):>12} {is_win:>8}')
    
    for _, r in dn_pnl_final.iterrows():
        side = 'DOWN'
        entry = r['od83_dn_entry']
        exit_p = r['od83_dn_exit']
        pnl = r['od83_dn_pnl']
        reason = int(r['od83_dn_r'])
        total_pnl += pnl
        is_win = 'WIN' if pnl > 0 else 'LOSS'
        if pnl > 0: wins += 1
        else: losses += 1
        print(f'  {side:>5} {entry:>8.4f} {exit_p:>8.4f} {7:>6} {pnl:>+10.4f} {reasons.get(reason,str(reason)):>12} {is_win:>8}')
    
    print(f'{"─"*60}')
    total_trades = wins + losses
    win_rate = 100 * wins / max(1, total_trades)
    print(f'  TOTAL PnL: {total_pnl:+.4f} | Trades: {total_trades} | Wins: {wins} | Losses: {losses} | Win%: {win_rate:.1f}%')
    print(f'  Budget: \$20 | Return: {100*total_pnl/20:+.1f}%')
    
    # Visual: PnL evolution during active positions
    active_up = df[df['od83_up'] == 2]
    active_dn = df[df['od83_dn'] == 2]
    if len(active_up) > 0 or len(active_dn) > 0:
        fig, ax = plt.subplots(figsize=(14, 4))
        if len(active_up) > 0:
            ax.plot(active_up['ts'], active_up['od83_up_pnl'], 'g-', lw=1.5, label='UP PnL')
        if len(active_dn) > 0:
            ax.plot(active_dn['ts'], active_dn['od83_dn_pnl'], 'r-', lw=1.5, label='DOWN PnL')
        ax.axhline(0, color='gray', ls='--', alpha=0.5)
        ax.set_ylabel('Paper PnL ($)')
        ax.set_xlabel('Time')
        ax.set_title(f'Odiseo 83 PnL Evolution — {mode} Mode')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# Session summary line
print(f'\n{"="*70}')
if n_entries > 0:
    print(f'  Session result: {total_pnl:+.4f} ({100*total_pnl/20:+.1f}%) in {mode}')
else:
    print(f'  Session result: NO TRADES — Odiseo stayed in WATCHING')
    filters_on = int(df['od83_filters'].max()) if 'od83_filters' in df.columns else 0
    print(f'  Filters active: {filters_on} (0=RAW, 63=all on)')
print(f'{"="*70}')


In [ ]:
# @title Filter Impact Analysis — ¿Qué filtros bloquean más?
import numpy as np

print('=' * 70)
print('  FILTER IMPACT ANALYSIS')
print('=' * 70)

# Simulate each filter in isolation
trigger_mask = (df['clob_trade_up'] >= 0.83) | (df['clob_trade_dn'] >= 0.83)
triggers = df[trigger_mask]
print(f'\nTotal triggers: {len(triggers)}')

filters = [
    ('frozen_market', df['tick_gap_ms'] > 3000, 'Gap > 3s'),
    ('spread_health', abs(df['clob_trade_up'] - df['clob_trade_dn']) > 0.95, '|Trade spread| > 0.95'),
    ('flash_dump', df['dump_score'] >= 2, 'Dump score >= 2'),
    ('min_volume', (df['bid_vol'] < 500) | (df['ask_vol'] < 500), 'Vol < 500'),
    ('btc_trend_up', (df['clob_trade_up'] >= 0.83) & (df['btc_vel'] < -0.5), 'UP entry + BTC falling'),
    ('btc_trend_dn', (df['clob_trade_dn'] >= 0.83) & (df['btc_vel'] > 0.5), 'DN entry + BTC rising'),
    ('reversal_risk', df['reversal_score'] >= 2, 'Reversal >= 2'),
    ('spoof_protect', df['spoof'] == 1, 'Spoof detected'),
    ('ask_wall', df['ask_wall'] == 1, 'Ask wall active'),
]

results = []
for name, condition, desc in filters:
    if condition.name is None:
        continue
    # Apply both trigger + this filter condition
    blocked = trigger_mask & condition
    n_blocked = blocked.sum()
    # Of the triggers, how many would pass?
    would_pass = trigger_mask & ~condition
    n_pass = would_pass.sum()
    results.append({
        'filter': name,
        'desc': desc,
        'blocked': int(n_blocked),
        'blocked_pct': 100 * n_blocked / max(1, trigger_mask.sum()),
        'pass': int(n_pass),
    })

# Sort by most blocking
results.sort(key=lambda x: x['blocked'], reverse=True)

print(f'\n  {"Filter":<20} {"Condition":<25} {"Blocked":>10} {"%":>8} {"Pass":>8}')
print('  ' + '-' * 72)
for r in results:
    bar = '█' * int(r['blocked_pct'] / 5)
    print(f'  {r["filter"]:<20} {r["desc"]:<25} {r["blocked"]:>10} {r["blocked_pct"]:>7.1f}% {r["pass"]:>8} {bar}')

# Visual: waterfall
fig, ax = plt.subplots(figsize=(10, 5))
names = [r['filter'] for r in results]
blocked_vals = [r['blocked'] for r in results]
colors = ['#ef4444' if v > 100 else '#f97316' if v > 10 else '#eab308' for v in blocked_vals]
ax.barh(names, blocked_vals, color=colors, alpha=0.7)
ax.set_xlabel('Triggers blocked')
ax.set_title(f'Filter Impact — {len(triggers)} total triggers')
for i, v in enumerate(blocked_vals):
    if v > 0:
        ax.text(v + 1, i, str(v), va='center', fontsize=9, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Correlation: which indicators predict entries?
print(f'\n{"="*70}')
print(f'  INDICATOR CORRELATION WITH TRADING ACTIVITY')
print(f'{"="*70}')

if 'od83_up' in df.columns:
    # Create activity indicator: 0=idle, 1=WATCH, 2=ACTIVE
    df['activity'] = df[['od83_up', 'od83_dn']].max(axis=1)
    
    indicators = ['btc_vel', 'btc_acel', 'clob_trade_up', 'clob_trade_dn',
                  'dump_score', 'reversal_score', 'tick_gap_ms',
                  'clob_trade_count_up', 'clob_trade_count_dn',
                  'imbalance', 'bid_vol', 'ask_vol']
    
    present = [c for c in indicators if c in df.columns]
    print(f'\n  {"Indicator":<25} {"Corr w/activity":>18} {"Corr w/clob_up":>18}')
    print('  ' + '-' * 62)
    for col in present:
        corr_act = df[col].corr(df['activity']) if df[col].dtype in ['float64','int64'] else 0
        corr_up = df[col].corr(df['clob_trade_up']) if df[col].dtype in ['float64','int64'] and 'clob_trade_up' in df.columns else 0
        act_s = '↑' if corr_act > 0.1 else '↓' if corr_act < -0.1 else ' '
        up_s = '↑' if corr_up > 0.1 else '↓' if corr_up < -0.1 else ' '
        print(f'  {col:<25} {act_s} {corr_act:>16.4f} {up_s} {corr_up:>16.4f}')
    
    print(f'\n  ↑ = positive correlation (indicator rises → more activity / higher UP price)')
    print(f'  ↓ = negative correlation')


In [ ]:
# @title 7. Resumen final por minuto
print('=' * 90)
print(f'  RESUMEN — Sesión #{meta["session_id"]}')
print('=' * 90)

btc_d = meta['btc_end'] - meta['btc_start']
print(f'\n  BTC: ${meta["btc_start"]:,.0f} → ${meta["btc_end"]:,.0f} ({btc_d:+,.0f} USD) | Outcome: {meta["outcome"]}')
print(f'  Build: {meta.get("build","?")} | Ticks: {len(df)}')

# Odiseo
up_act = (df['od83_up'] == 2).sum()
dn_act = (df['od83_dn'] == 2).sum()
triggers = ((df['clob_trade_up'] >= 0.83) | (df['clob_trade_dn'] >= 0.83)).sum()

print(f'\n  Odiseo 83:')
print(f'    Triggers (trade>=0.83): {triggers}')
print(f'    UP ACTIVE: {up_act} ticks | DOWN ACTIVE: {dn_act} ticks')

# Trade window
print(f'\n  Trade Window:')
print(f'    UP count max: {df["clob_trade_count_up"].max()}')
print(f'    DN count max: {df["clob_trade_count_dn"].max()}')
print(f'    UP avg price: {df.loc[df["clob_trade_up"]>0, "clob_trade_up"].mean():.4f}')
print(f'    DN avg price: {df.loc[df["clob_trade_dn"]>0, "clob_trade_dn"].mean():.4f}')

# Risk
print(f'\n  Riesgo:')
print(f'    dump_score >=2: {(df["dump_score"]>=2).sum()} ticks')
print(f'    reversal_score >=2: {(df["reversal_score"]>=2).sum()} ticks')
print(f'    tick_gap > 2s: {(df["tick_gap_ms"]>2000).sum()} ticks')
print(f'    btc_vel abs avg: {df["btc_vel"].abs().mean():.2f} USD/s')
print(f'    btc_acel abs avg: {df["btc_acel"].abs().mean():.2f} USD/s²')

# Tabla por minuto
print(f'\n  {"Minuto":<22} {"Ticks":>6} {"BTC Δ":>8} {"Trig UP":>8} {"Trig DN":>8} {"Dump":>5} {"Rev":>4} {"Cnt UP":>7} {"Cnt DN":>7}')
print('  ' + '-' * 80)
for _, row in minute_stats.iterrows():
    print(f'  {str(row["minute"]):<22} {int(row["ticks"]):>6} {row["btc_delta"]:>+8.0f} {int(row["triggers_up"]):>8} {int(row["triggers_dn"]):>8} {int(row["dump_max"]):>5} {int(row["reversal_max"]):>4} {int(row["trade_up_count"]):>7} {int(row["trade_dn_count"]):>7}')